# Chapter 4
## Numerical Solution of HH ODEs
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter04.ipynb)

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact

In [ ]:
def beta_n(v):
    return 0.125 * exp(-(v + 70.0) / 80.0)


def beta_m(v):
    return 4.0 * exp(-(v + 70.0) / 18.0)


def beta_h(v):
    return 1. / (exp(-(v + 40.0) / 10.0) + 1.0)


def alpha_n(v):
    return 0.01 * (-60.0 - v) / (exp((-60.0 - v) / 10.0) - 1.0)


def alpha_m(v):
    if np.abs(v + 45.0) > 1.0e-8:
        return (v + 45.0) / 10.0 / (1.0 - exp(-(v + 45.0) / 10.0))
    else:
        return 1.0


def alpha_h(v):
    return 0.07 * exp(-(v + 70) / 20)


def h_inf(v):
    return alpha_h(v) / (alpha_h(v) + beta_h(v))


def m_inf(v):
    return alpha_m(v) / (alpha_m(v) + beta_m(v))


def n_inf(v):
    return alpha_n(v) / (alpha_n(v) + beta_n(v))

## HH Limit Cycle

Phase portrait (\(v\) vs \(n\)) of the classical HH neuron, started away
from rest so the trajectory converges onto the periodic limit cycle.

In [ ]:
def simulate_hh_limit_cycle(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                             v_k=-82.0, v_na=45.0, v_l=-59.0,
                             i_ext=10.0, t_final=50.0, dt=0.01):
    def derivative(x0, t):
        v, m, n, h = x0
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
              - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l)) / c
        dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return [dv, dm, dn, dh]

    v0 = -50.0
    x0 = [v0, m_inf(v0), 0.4, 0.6]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0], sol[:, 2]


def plot_hh_limit_cycle(t, v, n):
    fig, ax = plt.subplots(1, figsize=(5, 5))
    ax.plot(v, n, lw=2, c="k")
    ax.set_xlim(-100, 50)
    ax.set_ylim([0, 1])
    ax.set_xlabel("v [mV]")
    ax.set_ylabel("n")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_limit_cycle(*simulate_hh_limit_cycle())

In [ ]:
interact(lambda i_ext=10.0: plot_hh_limit_cycle(*simulate_hh_limit_cycle(i_ext=i_ext)),
         i_ext=(0.0, 20.0, 0.5));

## HH Refractoriness

A second current pulse is injected on top of a sustained drive; the pulse's
effect depends on when it lands relative to the ongoing spike.

In [ ]:
def simulate_hh_refractoriness(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                                v_k=-82.0, v_na=45.0, v_l=-59.0,
                                i_ext=10.0, t_final=50.0, dt=0.01, pulse_onset=9.0):
    def derivative(x0, t, i_ext, pulse_onset):
        v, m, n, h = x0
        if pulse_onset < t < pulse_onset + 1:
            i_ext = i_ext + 20.0
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
              - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l)) / c
        dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return [dv, dm, dn, dh]

    v0 = -50.0
    x0 = [v0, m_inf(v0), 0.4, 0.6]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t, args=(i_ext, pulse_onset))
    return t, sol[:, 0]


def plot_hh_refractoriness(pulse_onsets=(500.0, 5.0, 9.0), **kwargs):
    fig, ax = plt.subplots(3, figsize=(7, 6), sharex=True)
    for i, pulse_onset in enumerate(pulse_onsets):
        t, v = simulate_hh_refractoriness(pulse_onset=pulse_onset, **kwargs)
        ax[i].plot(t, v, lw=2, c="k")
        ax[i].plot([pulse_onset, pulse_onset + 1], [-100, -100], lw=3, c="r")
        ax[i].set_ylim([-100, 50])
        ax[i].set_ylabel("v [mV]", fontsize=14)
        ax[i].tick_params(labelsize=14)
    ax[0].set_xlim(0, t[-1])
    ax[2].set_xlabel("time [ms]", fontsize=14)
    ax[0].set_yticks(range(-100, 100, 50))
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_refractoriness()

## HH Solution

Voltage trace and the three gating variables for a single sustained pulse.

In [ ]:
def simulate_hh_solution(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                          v_k=-82.0, v_na=45.0, v_l=-59.0,
                          i_ext=10.0, t_final=50.0, dt=0.01):
    def derivative(x0, t):
        v, m, n, h = x0
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
              - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l)) / c
        dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return [dv, dm, dn, dh]

    v0 = -50.0
    x0 = [v0, m_inf(v0), 0.4, 0.6]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0], sol[:, 1], sol[:, 2], sol[:, 3]


def plot_hh_solution(t, v, m, n, h):
    fig, ax = plt.subplots(2, figsize=(7, 6), sharex=True)
    ax[0].plot(t, v, lw=2, c="k")
    ax[1].plot(t, m, lw=2, label="m", c="b")
    ax[1].plot(t, h, lw=2, label="h", c="g")
    ax[1].plot(t, n, lw=2, label="n", c="r")
    ax[0].set_xlim(min(t), max(t))
    ax[0].set_ylim(-100, 50)
    ax[1].set_xlabel("time [ms]")
    ax[0].set_ylabel("v [mV]")
    ax[0].set_yticks(range(-100, 100, 50))
    ax[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_hh_solution(*simulate_hh_solution())

In [ ]:
interact(lambda i_ext=10.0: plot_hh_solution(*simulate_hh_solution(i_ext=i_ext)),
         i_ext=(0.0, 20.0, 0.5));